In [1]:
import pandas as pd
from sklearn.model_selection import train_test_split

In [2]:
train_df = pd.read_csv('train.csv')
train_df['created_date'] = pd.to_datetime(train_df['created_date'])

In [3]:
train_df['year'] = train_df['created_date'].dt.year
train_df['month'] = train_df['created_date'].dt.month
train_df['day'] = train_df['created_date'].dt.day

In [4]:
X_train, X_val, y_train, y_val = train_test_split(train_df.loc[:,train_df.columns!='label'],train_df.loc[:,'label'],random_state=42, train_size=.8)

In [5]:
X_train.shape[0]+X_val.shape[0]

198000

In [6]:
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

In [7]:
X_train['comment'] = X_train['comment'].fillna('')
X_val['comment'] = X_val['comment'].fillna('')

In [8]:
ohe_features = ['religion', 'gender', 'race']

In [9]:
encoder = OneHotEncoder(handle_unknown='ignore', sparse_output=False)

# 3. Create a pipeline that first imputes with 'none', then encodes
# We use a ColumnTransformer to apply OHE only to the specified columns
preprocessor = ColumnTransformer(
    transformers=[
        ('cat_ohe', Pipeline([
            ('imputer', SimpleImputer(strategy='constant', fill_value='none')),
            ('encoder', encoder)
        ]), ohe_features)
    ],
    remainder='passthrough' # Keeps other columns in the dataframe
)

In [10]:
X_train_vals = preprocessor.fit_transform(X_train)

In [11]:
X_train = pd.DataFrame(X_train_vals,columns=preprocessor.get_feature_names_out())

In [12]:
X_train.shape

(158400, 33)

In [13]:
X_val_vals = preprocessor.transform(X_val)

In [14]:
X_val = pd.DataFrame(X_val_vals,columns=preprocessor.get_feature_names_out())

In [15]:
from sklearn.feature_extraction.text import CountVectorizer

In [16]:
cv = CountVectorizer()

In [17]:
train_cv = cv.fit_transform(X_train['remainder__comment'])

In [18]:
val_cv = cv.transform(X_val['remainder__comment'])

In [19]:
X_train = X_train.drop(['remainder__created_date', 'remainder__comment'],axis=1)

In [20]:
X_val = X_val.drop(['remainder__created_date', 'remainder__comment'],axis=1)

In [23]:
X_train['remainder__disability'] = X_train['remainder__disability'].astype(int)

In [24]:
X_val['remainder__disability'] = X_val['remainder__disability'].astype(int)

In [32]:
X_train = X_train.astype('float32')
X_val = X_val.astype('float32')

In [33]:
from scipy.sparse import hstack, csr_matrix
sparse_X_train = hstack([csr_matrix(X_train.values),train_cv])
sparse_X_val = hstack([csr_matrix(X_val.values),val_cv])

In [34]:
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler(with_mean=False)
X_train_scaled = scaler.fit_transform(sparse_X_train)
X_val_scaled = scaler.transform(sparse_X_val)

In [35]:
X_train_scaled

<158400x149726 sparse matrix of type '<class 'numpy.float64'>'
	with 7767097 stored elements in Compressed Sparse Row format>